In [ ]:
# Import libraries and modules
import sys
import warnings
from pathlib import Path

# Add src directory to path
sys.path.append(str(Path('..').absolute()))

from src.config.constants import DATA_PATH, DATASET_NAMES, DATASET_INFO
from src.utils.helpers import setup_directories, load_dataset, print_section_header, AnalysisTimer
from src.analysis.basic_eda import BasicEDA
from src.analysis.advanced_analysis import AdvancedAnalysis

# Setup
warnings.filterwarnings('ignore')
results_path = setup_directories()

print_section_header("NASA TURBOFAN ENGINE RUL - CLEAN EDA")
print("Starting clean, modular analysis...")
print(f"Data path: {DATA_PATH}")
print(f"Results will be saved to: {results_path}")

In [ ]:
# Load all datasets
print_section_header("DATA LOADING")

datasets = {}

with AnalysisTimer("Loading all datasets"):
    for name in DATASET_NAMES:
        print(f"Loading {name}...")
        dataset = load_dataset(name, DATA_PATH)
        
        if dataset:
            datasets[name] = dataset
            train_shape = dataset['train'].shape
            test_shape = dataset['test'].shape
            print(f"  ✓ {name} - Train: {train_shape}, Test: {test_shape}")
        else:
            print(f"  ✗ Failed to load {name}")

print(f"\nSuccessfully loaded {len(datasets)}/{len(DATASET_NAMES)} datasets")

In [ ]:
# Basic Exploratory Data Analysis
print_section_header("BASIC EXPLORATORY DATA ANALYSIS")

# Initialize EDA analyzer
eda = BasicEDA(datasets)

# Dataset overview
with AnalysisTimer("Dataset overview analysis"):
    overview_df = eda.analyze_dataset_overview()
    print("\nDataset Overview:")
    print(overview_df.to_string(index=False))

# RUL distributions
with AnalysisTimer("RUL distribution analysis"):
    eda.plot_rul_distributions()

# Engine lifecycles for each dataset
with AnalysisTimer("Engine lifecycle analysis"):
    for dataset_name in DATASET_NAMES:
        eda.plot_engine_lifecycles(dataset_name, max_engines=5)

In [ ]:
# Sensor Analysis
print_section_header("SENSOR ANALYSIS")

# Analyze sensor variance and correlations for each dataset
sensor_results = {}
correlation_results = {}
trend_results = {}

for dataset_name in DATASET_NAMES:
    print(f"\nAnalyzing {dataset_name}...")
    
    with AnalysisTimer(f"Sensor variance analysis for {dataset_name}"):
        # Sensor variance analysis
        sensor_stats = eda.analyze_sensor_variance(dataset_name)
        low_variance_sensors = sensor_stats[sensor_stats['is_low_variance']]
        
        print(f"  Low variance sensors: {len(low_variance_sensors)}")
        if len(low_variance_sensors) > 0:
            print(f"  Examples: {', '.join(low_variance_sensors.index[:3])}")
        
        sensor_results[dataset_name] = sensor_stats
    
    with AnalysisTimer(f"Correlation analysis for {dataset_name}"):
        # Sensor correlations
        corr_matrix = eda.plot_sensor_correlations(dataset_name)
        correlation_results[dataset_name] = corr_matrix
    
    with AnalysisTimer(f"Degradation trend analysis for {dataset_name}"):
        # Degradation patterns
        eda.analyze_degradation_patterns(dataset_name)
        
        # Trend analysis
        trends = eda.calculate_degradation_trends(dataset_name)
        strong_indicators = trends[trends['degradation_indicator']]
        
        print(f"  Strong degradation indicators: {len(strong_indicators)}")
        if len(strong_indicators) > 0:
            top_indicators = strong_indicators.sort_values('correlation', key=abs, ascending=False).head(3)
            print(f"  Top 3: {', '.join(top_indicators.index)}")
        
        trend_results[dataset_name] = trends

In [ ]:
# Advanced Analysis
print_section_header("ADVANCED ANALYSIS")

# Initialize advanced analyzer
advanced = AdvancedAnalysis(datasets)

# Uncertainty analysis
uncertainty_results = {}
pca_results = {}
drift_results = {}

for dataset_name in DATASET_NAMES:
    print(f"\n🔬 Advanced analysis for {dataset_name}...")
    
    with AnalysisTimer(f"Uncertainty analysis for {dataset_name}"):
        uncertainty = advanced.analyze_uncertainty_baseline(dataset_name)
        uncertainty_results[dataset_name] = uncertainty
        
        n_conditions = len(uncertainty)
        avg_cv = np.mean([u['cv_lifespan'] for u in uncertainty.values()])
        print(f"  Operating conditions: {n_conditions}")
        print(f"  Average CV of lifespan: {avg_cv:.3f}")
    
    with AnalysisTimer(f"PCA analysis for {dataset_name}"):
        pca_result = advanced.perform_pca_analysis(dataset_name)
        pca_results[dataset_name] = pca_result
        
        # Print PCA insights
        explained_var = pca_result['explained_variance_ratio'][:3]
        cumulative_var = pca_result['cumulative_variance'][2]  # First 3 components
        print(f"  First 3 PCs explain {cumulative_var:.1%} of variance")
        print(f"  PC1: {explained_var[0]:.1%}, PC2: {explained_var[1]:.1%}, PC3: {explained_var[2]:.1%}")
    
    with AnalysisTimer(f"Data drift analysis for {dataset_name}"):
        drift_df = advanced.analyze_data_drift(dataset_name)
        drift_results[dataset_name] = drift_df
        
        n_drift_features = drift_df['potential_drift'].sum()
        print(f"  Features with potential drift: {n_drift_features}/{len(drift_df)}")
        
        if n_drift_features > 0:
            top_drift = drift_df[drift_df['potential_drift']].sort_values('ks_statistic', ascending=False).head(3)
            print(f"  Top drift features: {', '.join(top_drift.index)}")

In [ ]:
# Bootstrap Uncertainty Analysis
print_section_header("BOOTSTRAP UNCERTAINTY QUANTIFICATION")

import numpy as np

# Bootstrap analysis for sensor-RUL correlations
bootstrap_results = {}

for dataset_name in DATASET_NAMES:
    print(f"\n🎲 Bootstrap analysis for {dataset_name}...")
    
    with AnalysisTimer(f"Bootstrap correlations for {dataset_name}"):
        bootstrap_corr = advanced.bootstrap_sensor_correlations(dataset_name, n_bootstrap=100)
        bootstrap_results[dataset_name] = bootstrap_corr
        
        # Find most stable and unstable correlations
        stable_sensors = [s for s, data in bootstrap_corr.items() if data.get('is_stable', False)]
        unstable_sensors = [s for s, data in bootstrap_corr.items() if not data.get('is_stable', True)]
        
        print(f"  Stable correlations: {len(stable_sensors)}")
        print(f"  Unstable correlations: {len(unstable_sensors)}")
        
        # Show top 3 strongest correlations with confidence intervals
        correlations_sorted = sorted(bootstrap_corr.items(), 
                                   key=lambda x: abs(x[1]['mean']), reverse=True)
        
        print("  Top 3 strongest correlations (with 95% CI):")
        for i, (sensor, stats) in enumerate(correlations_sorted[:3]):
            mean_corr = stats['mean']
            ci_lower = stats['ci_lower']
            ci_upper = stats['ci_upper']
            print(f"    {i+1}. {sensor}: {mean_corr:.3f} [{ci_lower:.3f}, {ci_upper:.3f}]")

In [ ]:
# Summary and Key Findings
print_section_header("SUMMARY AND KEY FINDINGS")

print("📊 ANALYSIS COMPLETE - Key Insights:")
print("\n1. DATASET CHARACTERISTICS:")
for name in DATASET_NAMES:
    info = DATASET_INFO[name]
    train_engines = len(datasets[name]['train']['unit_id'].unique())
    avg_life = datasets[name]['train'].groupby('unit_id')['time_cycles'].max().mean()
    print(f"   • {name}: {train_engines} engines, avg life {avg_life:.0f} cycles - {info['description']}")

print("\n2. SENSOR INSIGHTS:")
for name in DATASET_NAMES:
    sensor_stats = sensor_results[name]
    trends = trend_results[name]
    
    low_var_count = sensor_stats['is_low_variance'].sum()
    strong_indicators = trends['degradation_indicator'].sum()
    
    print(f"   • {name}: {low_var_count} low-variance sensors, {strong_indicators} strong degradation indicators")

print("\n3. UNCERTAINTY FINDINGS:")
for name in DATASET_NAMES:
    uncertainty = uncertainty_results[name]
    n_conditions = len(uncertainty)
    avg_cv = np.mean([u['cv_lifespan'] for u in uncertainty.values()])
    
    print(f"   • {name}: {n_conditions} operating conditions, avg lifespan CV = {avg_cv:.3f}")

print("\n4. DATA QUALITY:")
for name in DATASET_NAMES:
    drift_df = drift_results[name]
    n_drift = drift_df['potential_drift'].sum()
    print(f"   • {name}: {n_drift} features show potential train/test drift")

print("\n🎯 RECOMMENDATIONS:")
print("   • Focus on sensors with strong, stable correlations with RUL")
print("   • Consider operating condition effects in model design")
print("   • Address data drift through domain adaptation techniques")
print("   • Use uncertainty quantification for confidence bounds")
print("   • Apply dimensionality reduction (PCA shows good variance capture)")

print("\n✅ All analysis results saved to:", results_path.absolute())

In [ ]:
# NASA Turbofan Engine RUL - Clean EDA

## Overview
This notebook provides a clean, modular approach to exploratory data analysis for the NASA C-MAPSS turbofan engine dataset for Remaining Useful Life (RUL) prediction.

## Dataset Information:
- **FD001**: 100 train/100 test engines, 1 condition, 1 fault mode (HPC degradation)
- **FD002**: 260 train/259 test engines, 6 conditions, 1 fault mode (HPC degradation)  
- **FD003**: 100 train/100 test engines, 1 condition, 2 fault modes (HPC + Fan degradation)
- **FD004**: 248 train/249 test engines, 6 conditions, 2 fault modes (HPC + Fan degradation)

## Analysis Structure
1. **Basic EDA**: Dataset overview, RUL distributions, sensor correlations
2. **Advanced Analysis**: Uncertainty quantification, PCA, drift detection
3. **Statistical Analysis**: Bootstrap confidence intervals, significance testing
4. **Visualization**: Clean, publication-ready plots